In [1]:
import arcpy
from arcpy.sa import *
import os
import glob
import shutil
arcpy.env.overwriteOutput = True

In [2]:
raster_names = [
    "MAP",
    "MAT",
    "Elevation",
    "Slope",
    "Aspect",
    "TWI",
    "NDWI",
    "NDVI"
]

In [13]:
raster_basepaths = [
    "F:/databases/climate/prism_ppt_us_30s_2020_avg_30y/ppt_avg_30y_IL_LambertNAD1983",
    "F:/databases/climate/prism_tmean_us_30s_2020_avg_30y/tmean_avg_30y_IL_LambertNAD1983",
    "F:/wim_rasters/dem/IL_DEM_3m_LambertNAD1983",
    "F:/wim_rasters/slope/Slope_3m_LambertNAD1983",
    "F:/wim_rasters/aspect/Aspect_3m_LambertNAD1983",
    "F:/wim_rasters/twi/TWI_IL_Final_LambertNAD1983_Mosaic",
    "F:/wim_rasters/ndwi/NDWI_IL_LambertNAD1983_mosaic",
    "F:/wim_rasters/ndvi/NDVI_IL_Final_LambertNAD1983"
]
n_r = len(raster_basepaths)

In [14]:
in_rasters = []
out_rasters = []
for i in range(n_r):
    path = raster_basepaths[i]
    name = raster_names[i]
    out_rasters.append(os.path.normpath(os.path.join(path+"_aligned.tif")))
    in_rasters.append(os.path.normpath(os.path.join(path+".tif")))

In [15]:
in_rasters

['F:\\databases\\climate\\prism_ppt_us_30s_2020_avg_30y\\ppt_avg_30y_IL_LambertNAD1983.tif',
 'F:\\databases\\climate\\prism_tmean_us_30s_2020_avg_30y\\tmean_avg_30y_IL_LambertNAD1983.tif',
 'F:\\wim_rasters\\dem\\IL_DEM_3m_LambertNAD1983.tif',
 'F:\\wim_rasters\\slope\\Slope_3m_LambertNAD1983.tif',
 'F:\\wim_rasters\\aspect\\Aspect_3m_LambertNAD1983.tif',
 'F:\\wim_rasters\\twi\\TWI_IL_Final_LambertNAD1983_Mosaic.tif',
 'F:\\wim_rasters\\ndwi\\NDWI_IL_LambertNAD1983_mosaic.tif',
 'F:\\wim_rasters\\ndvi\\NDVI_IL_Final_LambertNAD1983.tif']

In [16]:
out_rasters

['F:\\databases\\climate\\prism_ppt_us_30s_2020_avg_30y\\ppt_avg_30y_IL_LambertNAD1983_aligned.tif',
 'F:\\databases\\climate\\prism_tmean_us_30s_2020_avg_30y\\tmean_avg_30y_IL_LambertNAD1983_aligned.tif',
 'F:\\wim_rasters\\dem\\IL_DEM_3m_LambertNAD1983_aligned.tif',
 'F:\\wim_rasters\\slope\\Slope_3m_LambertNAD1983_aligned.tif',
 'F:\\wim_rasters\\aspect\\Aspect_3m_LambertNAD1983_aligned.tif',
 'F:\\wim_rasters\\twi\\TWI_IL_Final_LambertNAD1983_Mosaic_aligned.tif',
 'F:\\wim_rasters\\ndwi\\NDWI_IL_LambertNAD1983_mosaic_aligned.tif',
 'F:\\wim_rasters\\ndvi\\NDVI_IL_Final_LambertNAD1983_aligned.tif']

# fill NoData streaks in NWDI

In [ ]:
ndwi_unfilled = 'F:\\wim_rasters\\ndwi\\NDWI_IL_LambertNAD1983.tif'
ndwi_filled = 'F:\\wim_rasters\\ndwi\\NDWI_IL_LambertNAD1983_filled.tif'

In [ ]:
filled = arcpy.sa.Con(IsNull(ndwi_unfilled), 
                      FocalStatistics(ndwi_unfilled, NbrRectangle(3,3), "MEAN"), 
                      ndwi_unfilled)
filled.save(ndwi_filled)

In [ ]:
arcpy.management.Mosaic(
    inputs='F:\\wim_rasters\\ndwi\\NDWI_IL_LambertNAD1983_filled.tif',
    target='F:\\wim_rasters\\ndwi\\NDWI_IL_LambertNAD1983_mosaic.tif',
    mosaic_type="FIRST",
    colormap="FIRST",
    background_value=None,
    nodata_value=None,
    onebit_to_eightbit="NONE",
    mosaicking_tolerance=0,
    MatchingMethod="NONE"
);

# Find raster with largest cell size

In [17]:
# Get cell size of each raster
cell_sizes = {}
for r_path in in_rasters:
    r = arcpy.Raster(r_path)
    cell_sizes[r_path] = r.meanCellWidth
    print(f"{os.path.basename(r_path):<30} {r.meanCellWidth}m")

ppt_avg_30y_IL_LambertNAD1983.tif 828.8118654485328m
tmean_avg_30y_IL_LambertNAD1983.tif 828.8118654485328m
IL_DEM_3m_LambertNAD1983.tif   3.0m
Slope_3m_LambertNAD1983.tif    3.0m
Aspect_3m_LambertNAD1983.tif   3.0m
TWI_IL_Final_LambertNAD1983_Mosaic.tif 3.0m
NDWI_IL_LambertNAD1983_mosaic.tif 3.000000000000001m
NDVI_IL_Final_LambertNAD1983.tif 3.0m


In [18]:
# Find coarsest
reference = max(cell_sizes, key=cell_sizes.get)
ref_r     = arcpy.Raster(reference)
ref_sr    = ref_r.spatialReference
ref_cell  = ref_r.meanCellWidth

print(f"\nReference raster:    {os.path.basename(reference)}")
print(f"Reference cell size: {cell_sizes[reference]}m")
print(f"Reference CRS:       {ref_r.spatialReference.name}")


Reference raster:    ppt_avg_30y_IL_LambertNAD1983.tif
Reference cell size: 828.8118654485328m
Reference CRS:       NAD_1983_Lambert_Conformal_Conic_m


# Resample and align all to reference

In [19]:
# Set snap raster to reference — forces grid alignment
arcpy.env.snapRaster             = reference
arcpy.env.outputCoordinateSystem = None
resample_method = "BILINEAR"  # continuous

In [20]:
# Resample all rasters to match reference
for i in range(n_r-2,n_r-1):
    raster_name  = raster_names[i]
    out_raster_path = out_rasters[i]
    in_raster_path = in_rasters[i]
    in_raster     = arcpy.Raster(in_raster_path)

    # skip if already matches reference
    if (in_raster.meanCellWidth  == ref_cell and
        in_raster.spatialReference.factoryCode == ref_sr.factoryCode):
        print(f"Already aligned: {raster_name}")
        shutil.copy2(in_raster_path, out_raster_path)
        continue

    print(f"Resampling: {raster_name} ({in_raster.meanCellWidth}m → {ref_cell}m)")

    # project raster to match reference CRS if needed
    if in_raster.spatialReference.factoryCode != ref_sr.factoryCode:
        print(f"Reprojecting CRS: {in_raster.spatialReference.name} → {ref_sr.name}")
        arcpy.management.ProjectRaster(
            in_raster=in_raster_path,
            out_raster=out_raster_path,
            out_coor_system=ref_sr,
            resampling_type=resample_method,
            cell_size=str(ref_cell)
        )
    else:
        # same CRS — just resample
        arcpy.management.Resample(
            in_raster=in_raster_path,
            out_raster=out_raster_path,
            cell_size=str(ref_cell),
            resampling_type=resample_method
        )

Resampling: NDWI (3.000000000000001m → 828.8118654485328m)


# Copy all to 64 bit rasters

In [26]:
out_rasters

['F:\\databases\\climate\\prism_ppt_us_30s_2020_avg_30y\\ppt_avg_30y_IL_LambertNAD1983_aligned.tif',
 'F:\\databases\\climate\\prism_tmean_us_30s_2020_avg_30y\\tmean_avg_30y_IL_LambertNAD1983_aligned.tif',
 'F:\\wim_rasters\\dem\\IL_DEM_3m_LambertNAD1983_aligned.tif',
 'F:\\wim_rasters\\slope\\Slope_3m_LambertNAD1983_aligned.tif',
 'F:\\wim_rasters\\aspect\\Aspect_3m_LambertNAD1983_aligned.tif',
 'F:\\wim_rasters\\twi\\TWI_IL_Final_LambertNAD1983_Mosaic_aligned.tif',
 'F:\\wim_rasters\\ndwi\\NDWI_IL_LambertNAD1983_mosaic_aligned.tif',
 'F:\\wim_rasters\\ndvi\\NDVI_IL_Final_LambertNAD1983_aligned.tif']

In [27]:
for i in range(n_r):
    arcpy.management.CopyRaster(
        in_raster = out_rasters[i],
        out_rasterdataset = raster_basepaths[i] + "_aligned_64bit.tif",
        pixel_type="64_BIT"
    )